In [1]:
import calendar  
import glob  
import io  
import os  
import re
import msoffcrypto  
import pandas as pd  
from dotenv import load_dotenv  
import warnings
from msoffcrypto.exceptions import FileFormatError, InvalidKeyError

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [13]:
# read excel sheet with a password
def read_protected_excel(file_path, password, sheet_names, skiprows_list):  
    """Read multiple sheets from an Excel file that may or may not be password-protected."""  
    with open(file_path, 'rb') as f:  
        raw = io.BytesIO(f.read())

    workbook = raw  
    encrypted = False

    try:  
        raw.seek(0)  
        office_file = msoffcrypto.OfficeFile(raw)  
        if office_file.is_encrypted():  
            encrypted = True  
            decrypted = io.BytesIO()  
            office_file.load_key(password=password)  
            office_file.decrypt(decrypted)  
            workbook = decrypted  
    except FileFormatError:  
        workbook = raw  
    except InvalidKeyError:  
        raise RuntimeError(f"Wrong password for {os.path.basename(file_path)}")

    print(f"    ({'encrypted' if encrypted else 'not encrypted'})")

    dfs = []  
    for sheet_name, skiprows in zip(sheet_names, skiprows_list):  
        workbook.seek(0)  
        df = pd.read_excel(workbook, sheet_name=sheet_name, skiprows=skiprows)  
        dfs.append(df)

    return dfs
  
# set the file path for manpower files
base_paths = ['L:/2025 Pilar Plant Files/2025 Manpower Detail/',  
              'L:/2026 Pilar Plant Files/2026 Manpower Detail/']  
load_dotenv()  
password = os.getenv("SM_REPORTS_PW")  
sheet_name = 'PPE FTEs'  
skiprows = 8

all_ppe_ftes = []
keep_dates = pd.to_datetime(['2026-05-31', '2026-10-31', '2025-11-15', '2025-11-30', '2025-12-15'])


for base_path in base_paths:  
    for file_path in glob.glob(os.path.join(base_path, '*.xlsm')):  
        filename = os.path.basename(file_path)  
        if filename.startswith('~$'):   # skip Excel lock files  
            continue

        date = pd.to_datetime(filename[-15:-5], format='%Y-%m-%d', errors='coerce')  
        site = filename.split(' ')[0].strip()

        # skip bad dates, the 15th, and month-end  
        if pd.isna(date):  
            continue 
        if date not in keep_dates and (date.day == 15 or date.is_month_end):  
            continue  

        print(f"Reading {site} | {date.date()}")  
        try:  
            df = read_protected_excel(file_path, password, [sheet_name], [skiprows])[0]  
            df['Pay_Type'] = 'Biweekly'  
            df['Report_Date'] = date  
            df['BU'] = site  
            all_ppe_ftes.append(df)  
        except Exception as e:  
            print(f"  ERROR: {e}")

ppe_ftes_combined = pd.concat(all_ppe_ftes, ignore_index=True)

ppe_ftes_combined.columns = (  
    ppe_ftes_combined.columns.astype(str)  
    .str.strip()  
    .str.replace(r'[^a-zA-Z0-9]', '_', regex=True)  
    .str.replace(r'_+', '_', regex=True)  
    .str.strip('_')  
    .str.lower()  
)

print(f"\nPPE FTEs: {ppe_ftes_combined.shape}")  
print(f"Sites found: {ppe_ftes_combined['bu'].unique()}")  

Reading GRTLH | 2025-11-30
    (encrypted)
  ERROR: Worksheet named 'PPE FTEs' not found
Reading GVCCC | 2025-01-11
    (encrypted)
Reading GVCCC | 2025-01-25
    (encrypted)
Reading GVCCC | 2025-02-08
    (encrypted)
Reading GVCCC | 2025-02-22
    (encrypted)
Reading GVCCC | 2025-03-08
    (encrypted)
Reading GVCCC | 2025-03-22
    (encrypted)
Reading GVCCC | 2025-04-05
    (not encrypted)
Reading GVCCC | 2025-04-19
    (encrypted)
Reading GVCCC | 2025-05-03
    (encrypted)
Reading GVCCC | 2025-05-17
    (encrypted)
Reading GVCCC | 2025-07-12
    (encrypted)
Reading GVCCC | 2025-07-26
    (encrypted)
Reading GVCCC | 2025-08-09
    (encrypted)
Reading GVCCC | 2025-08-23
    (encrypted)
Reading GVCCC | 2025-09-06
    (encrypted)
Reading GVCCC | 2025-09-20
    (encrypted)
Reading GVCCC | 2025-10-04
    (encrypted)
Reading GVCCC | 2025-10-18
    (encrypted)
Reading GVCCC | 2025-11-01
    (encrypted)
Reading GVCCC | 2025-11-15
    (encrypted)
Reading GVCCC | 2025-11-30
    (encrypted)
Read

In [27]:
# function to clean ftes tabs
def clean_ftes(sheet):
    # remove subtotal rowa
    sheet_c = sheet[~sheet['deptid'].str.contains('total', case=False, na=False)]

    # forward fill department ids and names
    sheet_c['deptid'] = sheet_c['deptid'].ffill()
    sheet_c['department_description'] = sheet_c['department_description'].ffill()

    # remove department ids that are blank
    sheet_n = sheet_c[sheet_c['deptid'] != '(blank)']
    # set type to integer
    sheet_n[['deptid', 'job_code']] = sheet_n[['deptid', 'job_code']].astype('Int64') 

    # fix duplicate column names  
    cols = pd.Series(sheet_n.columns)  
    for dup in cols[cols.duplicated()].unique():  
        count = 0  
        for i in range(len(cols)):  
            if cols[i] == dup:  
                count += 1  
                if count > 1:  
                    cols[i] = f"{dup}_{count}"  
    sheet_n.columns = cols
    return sheet_n

In [36]:
# clean ppe ftes
ppe_ftes_clean = clean_ftes(ppe_ftes_combined)

# --- budget ---
# select budget related columns
ppe_ftes_budget = ppe_ftes_clean[['report_date', 'bu', 'deptid', 'department_description', 'job_code', 'job_code_description', \
                                  'regular_non_prod', 'overtime_2', 'holiday_wkd_2', 'total_2']]

# create fte and type columns 
ppe_ftes_budget['reg_ftes'] = ppe_ftes_budget['total_2'] - ppe_ftes_budget['overtime_2']  
ppe_ftes_budget['ot_ftes'] = ppe_ftes_budget['overtime_2']  
ppe_ftes_budget['total_ftes'] = ppe_ftes_budget['total_2']
ppe_ftes_budget['type'] = 'Budget'

# select columns to keep
ppe_ftes_b = ppe_ftes_budget[['type', 'report_date', 'bu', 'deptid', 'department_description', 'job_code', 'job_code_description', \
                                  'reg_ftes', 'ot_ftes', 'total_ftes']]

# --- actual ---
ppe_ftes_actual = ppe_ftes_clean[['report_date', 'bu', 'deptid', 'department_description', 'job_code', 'job_code_description', \
                                  'regular', 'overtime', 'holiday_wkd', 'non_prod', 'total']]

# create fte and type columns
ppe_ftes_actual['reg_ftes'] = ppe_ftes_actual['total'] - ppe_ftes_actual['overtime']  
ppe_ftes_actual['ot_ftes'] = ppe_ftes_actual['overtime']  
ppe_ftes_actual['total_ftes'] = ppe_ftes_actual['total']
ppe_ftes_actual['type'] = 'Actual'

# select columns to keep
ppe_ftes_a = ppe_ftes_actual[['type', 'report_date', 'bu',  'deptid', 'department_description', 'job_code', 'job_code_description', \
                                  'reg_ftes', 'ot_ftes', 'total_ftes']].drop_duplicates()

# check totals
display(ppe_ftes_a.groupby(['bu', 'report_date'])[['ot_ftes']].sum())


ot = ppe_ftes_a[ppe_ftes_a['report_date'] >= '2025-11-01'].groupby(['report_date', 'bu', 'type', 'deptid', 'department_description'])[['ot_ftes']].sum().reset_index()


display(ot)

ot_ftes
bu    report_date         
GVCCC 2025-01-11     12.43
      2025-01-25     13.80
      2025-02-08     13.55
      2025-02-22     13.18
      2025-03-08     14.39
...                    ...
MEETH 2026-09-05     13.61
      2026-09-19     16.19
      2026-10-03     14.28
      2026-10-17     14.50
      2026-10-31     13.48

[135 rows x 1 columns]

,report_date,bu,type,deptid,department_description,ot_ftes
0,2025-11-01,GVCCC,Actual,74002500,Administration,0.05
1,2025-11-01,GVCCC,Actual,74002501,Materials Ops,0.28
2,2025-11-01,GVCCC,Actual,74002502,Access Services,3.28
3,2025-11-01,GVCCC,Actual,74002503,Environmental Services,3.61
4,2025-11-01,GVCCC,Actual,74002505,Radiology,1.94
...,...,...,...,...,...,...
5601,2026-10-31,MEETH,Actual,29690080,Sup Svc - Engineerg And Maint,1.89
5602,2026-10-31,MEETH,Actual,29691010,Sup Svc-Dietary,0.05
5603,2026-10-31,MEETH,Actual,29692010,Sup Svc-Security,0.00
5604,2026-10-31,MEETH,Actual,29693000,Sup Svc-Laundry and Linen,0.11


In [40]:
file_name = 'ot_ftes_251101_261031_v2'

with pd.ExcelWriter(f'C:/Users/kbixby/OneDrive - Northwell Health/Documents/FTE/{file_name}.xlsx', engine='openpyxl') as writer: 
    ot.to_excel(writer, sheet_name='FTEs', index=False)  

print(f"Written to {file_name}.xlsx")

Written to ot_ftes_251101_261031_v2.xlsx
